# Basic agent 1

In [59]:
import os
from getpass import getpass
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI 
import asyncio


In [60]:

client = AsyncOpenAI(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",  # dummy key (LM Studio doesn't require real one
    )

In [63]:
agent = Agent(
    name="my_agent",
    instructions="You're a helpful assistant",
    model=OpenAIChatCompletionsModel(  # wrapping model name + client together
        model="google/gemma-3-12b",
        openai_client=client  # passing the client here
    ),
)

OpenAI gives us three methods for running our agent, all via a Runner class — those methods are:

- Runner.run() which runs in async.
- Runner.run_sync() which runs in sync.
- Runner.run_streamed() which runs in async and streams the response back to us.

In [64]:
# Runner.run was not working in the python file properly
#but in notebook it works fine. So I switched to Runner.run_sync() in the python file and it works fine.
#but why? 
# Runner.run() is an async function, so it returns a coroutine. 
# In a regular Python script, you need to use asyncio.run() to execute the coroutine and get the result. 
# In a Jupyter notebook, you can directly await the coroutine without needing to wrap it in asyncio.run(), 
# which is why it works fine there.
result = await Runner.run( 
    starting_agent=agent, # passing the agent to starting_agent
    input="What is the capital of France?"
)

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [65]:
result.final_output #here it is providing the final output of the agent after processing the input 
# and any intermediate steps.

'The capital of France is **Paris**. 🇫🇷\n'

In most scenarios we'll likely want to be using method (3), ie running async and streaming tokens.
To do this we need to write a little more code to handle the async streaming and print the tokens as they're returned.

- First, we create a RunResultStreaming object by calling Runner.run_streamed(...), we then asynchronously iterate through the streamed events returned by our LLM using the response.stream_events() method:

In [44]:
# here we are using Runner.run_streamed() which runs in async and streams the response back to us.
# This is useful when you want to get intermediate outputs from the agent as it processes the input,
# without having to wait for the entire response to be generated.
# we can use run_streamed() for long-running tasks or when we want to display intermediate results 
# in real-time.
final_text = ""
response = Runner.run_streamed(
    starting_agent=agent,
    input="what is the capital of France?"
)
async for event in response.stream_events():
    print(event) # this will print each event as it comes in, including intermediate outputs and the final output
    if event.type == "response.output_text.delta":
        final_text += event.delta # this will accumulate the text output as it comes in, basically the delta is the new text that has been generated since the last event, 
        #so by adding it to final_text we get the full output as it is being generated
    if event.type == "response.completed":
        print("\nFinal Output:", final_text) # this will print the final output once the response is complete

AgentUpdatedStreamEvent(new_agent=Agent(name='my_agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions="You're a helpful assistant", prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x00000166204B1E50>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), type='agent_updated_stream_event')


OPENAI_API_KEY is not set, skipping trace export


RawResponsesStreamEvent(data=ResponseCreatedEvent(response=Response(id='__fake_id__', created_at=1775140968.7711556, error=None, incomplete_details=None, instructions=None, metadata=None, model='mistralai/mistral-nemo-instruct-2407', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='auto', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created'), type='raw_response_event')
RawResponsesStreamEvent(data=ResponseOutputItemAddedEvent(item=ResponseOutputMessage(id='__fake_id__', content=[], role='assistant', status='in_progress', type='message', phase=None, provider_data={'model': 'mistralai/mistral-nemo-instruct-24

In [45]:
response.final_output

"The capital of France is Paris. It's known for its iconic landmarks like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. Paris has been the capital since 508 AD and is also one of the most populous cities in Europe, with over 2 million people living within its arrondissements (districts)."

#### we can filter out the event types to find out raw tokens like so:

In [66]:
from openai.types.responses import ResponseTextDeltaEvent

# we do need to reinitialize our runner before re-executing
response = Runner.run_streamed(
    starting_agent=agent,
    input="tell me a short story (not more than 20 words)"
)

# here the ResponseTextDeltaEvent is a specific type of event that contains a delta of text output from the agent.
# and using that loop we can print the text output as it comes in, without waiting for the entire response to be generated.
async for event in response.stream_events():
    if event.type == "raw_response_event" and \
        isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

OPENAI_API_KEY is not set, skipping trace export


The old lighthouse keeper vanished, leaving only a single, unanswered message in the logbook: "They came from below."

OPENAI_API_KEY is not set, skipping trace export


In [47]:
from openai.types.responses import ResponseTextDeltaEvent

# we do need to reinitialize our runner before re-executing
response = Runner.run_streamed(
    starting_agent=agent,
    input="what if i delete you?"
)

# here the ResponseTextDeltaEvent is a specific type of event that contains a delta of text output from the agent.
# and using that loop we can print the text output as it comes in, without waiting for the entire response to be generated.
async for event in response.stream_events():
    if event.type == "raw_response_event" and \
        isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

If you delete me, our conversation will end

OPENAI_API_KEY is not set, skipping trace export


 and I'll no longer be able to assist you. However, when you come back or start a new session, a new instance of me will be created, and we can start fresh. Don't worry, I won't take it personally! 😊 Just remember that any context or information from this session will be lost if you delete me.

# Tools:
#### Let's take a look at the tools we can implement in the OpenAI sdk:

- OpenAI included function tools as a key feature in their Agents SDK announcement. After turning everyone away from using function calling to instead use tool calling, OpenAI have now decided that an LLM deciding to execute some code will be called "function tools".

- To use function tools in Agents SDK we simply decorate a function with the **@function_tool** decorator like so:

In [67]:
from agents import function_tool

# here the function_tool decorator is used to register this function as a tool that the agent can use.
@function_tool 
def multiply(a: float, b: float) -> float:
    '''Multiplies two numbers together.'''
    return a * b

Note that we have taken extra care to include a clear and descriptive function name, relatively clear parameter names, type annotations for both input parameters and expected output, and a natural language docstring that will be fed to the LLM and explain to it what this tool does.

To run our agent with tools we simply pass our new tool into the tools parameter during Agent initialization.

In [68]:
from agents import ModelSettings


agent = Agent(
    name="my_agent",
    instructions=(
        "You're a helpful assistant, remember to always "
        "use the provided tools whenever possible. Do not "
        "rely on your own knowledge too much and instead "
        "use your tools to help you answer queries."
    ),
    model=OpenAIChatCompletionsModel(  # wrapping model name + client together
        model="google/gemma-3-12b",
        openai_client=client  # passing the client here
    ),
    tools=[multiply],  # registering the multiply function as a tool the agent can use
    model_settings=ModelSettings(tool_choice="required")
) 

In [69]:
response = Runner.run_streamed(
    starting_agent=agent,
    input="what is 28.4854 multiplied by 348.4854?"
)

async for event in response.stream_events():
    print(event)

AgentUpdatedStreamEvent(new_agent=Agent(name='my_agent', handoff_description=None, tools=[FunctionTool(name='multiply', description='Multiplies two numbers together.', params_json_schema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'multiply_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000166239851D0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)], mcp_servers=[], mcp_config={}, instructions="You're a helpful assistant, remember to always use the provided tools whenever possible. Do not rely on your own knowledge too much and instead use your tools to help you answer queries.", prompt=None, handoffs=[], model=<agents.models.openai_chat

OPENAI_API_KEY is not set, skipping trace export


RawResponsesStreamEvent(data=ResponseCreatedEvent(response=Response(id='__fake_id__', created_at=1775142122.9545467, error=None, incomplete_details=None, instructions=None, metadata=None, model='google/gemma-3-12b', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='required', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created'), type='raw_response_event')
RawResponsesStreamEvent(data=ResponseOutputItemAddedEvent(item=ResponseFunctionToolCall(arguments='', call_id='412916326', name='multiply', type='function_call', id='__fake_id__', namespace=None, status=None, provider_data={'model': 'google/gemma-3-12b', '

OPENAI_API_KEY is not set, skipping trace export


RawResponsesStreamEvent(data=ResponseCreatedEvent(response=Response(id='__fake_id__', created_at=1775142156.5528686, error=None, incomplete_details=None, instructions=None, metadata=None, model='google/gemma-3-12b', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='auto', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created'), type='raw_response_event')
RawResponsesStreamEvent(data=ResponseOutputItemAddedEvent(item=ResponseOutputMessage(id='__fake_id__', content=[], role='assistant', status='in_progress', type='message', phase=None, provider_data={'model': 'google/gemma-3-12b', 'response_id': 'chatcmpl-ygog0y

OPENAI_API_KEY is not set, skipping trace export
